# Web Agent Action Prediction — Colab Pipeline

**최신화: 2026-05-12 (DPO Hard Negative v2 반영)**  
**Drive 없는 팀 계정 버전** — 파일 직접 업로드 + HuggingFace Hub 체크포인트

### 파이프라인 구조 (0512 기준)
```
workflow HTML  → build_prompt()          → LLM 단발 추론
real_web HTML  → Step 1 Grounding        → action_desc 수집
               → Round 1 (3그룹 × 5개)  → survivor 선출
               → Round 2 (survivors)    → 최종 후보
               → (no-survivor) B1       → 전체 15개 LLM 재추론
               → enforce_consistency()  → submission.csv
```

**DPO 학습 업데이트 사항**: `src/preprocess.py`에 적용된 `_is_false_negative` 필터와 하이브리드 정규화 스코어링이 반영된 `my_code.zip`을 업로드해야 합니다.

### 세션 시작 시 준비할 파일
- `my_code.zip` — `src/` 4개 파일 압축 (preprocess / retrieval / train / inference)
- `train.csv`, `test.csv`, `somenna_submission.csv`

### 실행 순서
1→2→3(선택)→4→5→6(선택)→**7 학습**→8 결과 확인→**9 추론**→10 리포트→**11 점검·다운로드**

## 1. GPU 확인

In [ ]:
import subprocess
gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader']).decode().strip()
print('GPU:', gpu)
assert any(g in gpu for g in ['T4','A100','L4','V100']), f'지원하지 않는 GPU: {gpu}'
!nvidia-smi

## 2. 패키지 설치

- `lxml`: BeautifulSoup 파서 (html.parser 대비 5~10배 빠름)
- `sentence-transformers` 불필요 (USE_RETRIEVAL=False, rerank 제거됨)

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "trl>=0.21" peft accelerate bitsandbytes
!pip install pandas tqdm scikit-learn
!pip install lxml
!pip install huggingface_hub
print('설치 완료')

## 3. HuggingFace 로그인 (선택 — 체크포인트 자동 저장용)

런타임이 꺼져도 체크포인트를 유지하려면 실행. 필요 없으면 건너뜀.  
**준비**: https://huggingface.co/settings/tokens 에서 `Write` 권한 토큰 발급

In [ ]:
USE_HF_HUB = True   # HF Hub 쓰지 않으려면 False

if USE_HF_HUB:
    from huggingface_hub import login
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = input('HuggingFace 토큰 입력: ').strip()

    HF_REPO = 'your-username/web-agent-lora'  # ★ 본인 repo로 변경
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f'HF Hub 연결됨: {HF_REPO}')
else:
    HF_TOKEN = None
    HF_REPO  = None
    print('HF Hub 미사용 — 학습 완료 후 즉시 추론하세요')

## 4. 파일 업로드

아래 셀 실행 후 파일 선택 창에서 한 번에 선택:
- `my_code.zip` — src/ 4개 파일
- `train.csv`, `test.csv`, `somenna_submission.csv`

In [ ]:
from google.colab import files
import os, zipfile, shutil

os.makedirs('/content/src',       exist_ok=True)
os.makedirs('/content/data',      exist_ok=True)
os.makedirs('/content/artifacts', exist_ok=True)

print('파일 선택: my_code.zip + train.csv + test.csv + somenna_submission.csv')
uploaded = files.upload()

with zipfile.ZipFile('my_code.zip') as z:
    z.extractall('/content/')

for f in ['train.csv', 'test.csv', 'somenna_submission.csv']:
    if os.path.exists(f):
        shutil.move(f, f'/content/data/{f}')

print('\nsrc/:')
!ls /content/src
print('\ndata/:')
!ls /content/data

## 5. GPU별 설정 자동 패치

| GPU | VRAM | DPO batch | DPO grad_accum | DPO lr | SFT batch | SFT lr | inf batch |
|-----|------|-------------|------------|-----------|----|-----------|
| **A100** | 40GB | 8 | 4 | 5e-5 | 32 | 2e-4 | 32 |
| L4  | 24GB | 4 | 8 | 5e-5 | 2 | 2e-4 | 8  |
| T4  | 16GB | 1 | 16| 5e-5 | 1 | 2e-4 | 4  |

> Step 1 Grounding + 토너먼트 2라운드로 추론 LLM 호출이 증가함.  
> A100 inf batch=32 기준 전체 추론 ~15분 예상.

In [ ]:
import subprocess, re

gpu_name = subprocess.check_output(
    ['nvidia-smi','--query-gpu=name','--format=csv,noheader']
).decode().strip()
is_a100 = 'A100' in gpu_name
is_l4   = 'L4'   in gpu_name
print(f'GPU: {gpu_name}')

tp = '/content/src/train.py'
ip = '/content/src/inference.py'

def patch(path, subs, flags=0):
    s = open(path).read()
    for pattern, repl in subs:
        s = re.sub(pattern, repl, s, flags=flags)
    open(path, 'w').write(s)

def patch_dpo_block(path, batch, accum, lr, steps, save_steps):
    s = open(path).read()
    pattern = r'(if USE_DPO:\n[\s\S]*?args = DPOConfig\([\s\S]*?\n\s*\),\n\s*\))'
    def repl(m):
        block = m.group(1)
        block = re.sub(r'per_device_train_batch_size\s*=\s*\d+', f'per_device_train_batch_size = {batch}', block)
        block = re.sub(r'gradient_accumulation_steps\s*=\s*\d+', f'gradient_accumulation_steps = {accum}', block)
        block = re.sub(r'learning_rate\s*=\s*[\d.e+-]+', f'learning_rate = {lr}', block)
        block = re.sub(r'max_steps\s*=\s*\d+', f'max_steps = {steps}', block)
        block = re.sub(r'save_steps\s*=\s*\d+', f'save_steps = {save_steps}', block)
        return block
    s, n = re.subn(pattern, repl, s, count=1)
    assert n == 1, f'DPOConfig block patch failed: {path}'
    open(path, 'w').write(s)

def patch_sft_block(path, batch, accum, lr, steps, save_steps):
    s = open(path).read()
    pattern = r'(else:\n\s*print\(f"\\nStarting SFT training[\s\S]*?args = SFTConfig\([\s\S]*?\n\s*\),\n\s*\))'
    def repl(m):
        block = m.group(1)
        block = re.sub(r'per_device_train_batch_size\s*=\s*\d+', f'per_device_train_batch_size = {batch}', block)
        block = re.sub(r'gradient_accumulation_steps\s*=\s*\d+', f'gradient_accumulation_steps = {accum}', block)
        block = re.sub(r'learning_rate\s*=\s*[\d.e+-]+', f'learning_rate = {lr}', block)
        block = re.sub(r'max_steps\s*=\s*\d+', f'max_steps = {steps}', block)
        block = re.sub(r'save_steps\s*=\s*\d+', f'save_steps = {save_steps}', block)
        block = re.sub(r'dataset_num_proc\s*=\s*\d+', 'dataset_num_proc = 4', block)
        return block
    s, n = re.subn(pattern, repl, s, count=1)
    assert n == 1, f'SFTConfig block patch failed: {path}'
    open(path, 'w').write(s)

if is_a100:
    patch_dpo_block(tp, batch=8, accum=4, lr='5e-5', steps=3000, save_steps=300)
    patch_sft_block(tp, batch=32, accum=1, lr='2e-4', steps=3000, save_steps=300)
    patch(ip, [(r'^BATCH_SIZE\s*=\s*\d+', 'BATCH_SIZE = 32')], flags=re.MULTILINE)
    print('A100 패치 완료')
elif is_l4:
    patch_dpo_block(tp, batch=4, accum=8, lr='5e-5', steps=2000, save_steps=200)
    patch_sft_block(tp, batch=2, accum=8, lr='2e-4', steps=1500, save_steps=150)
    patch(ip, [(r'^BATCH_SIZE\s*=\s*\d+', 'BATCH_SIZE = 8')], flags=re.MULTILINE)
    print('L4 패치 완료')
else:  # T4
    patch_dpo_block(tp, batch=1, accum=16, lr='5e-5', steps=1000, save_steps=100)
    patch_sft_block(tp, batch=1, accum=16, lr='2e-4', steps=1000, save_steps=100)
    patch(ip, [(r'^BATCH_SIZE\s*=\s*\d+', 'BATCH_SIZE = 4')], flags=re.MULTILINE)
    print('T4 패치 완료')

# 패치 결과 확인
import subprocess
for line in open(ip):
    if 'BATCH_SIZE' in line and '=' in line:
        print('inference.py:', line.strip())
        break
for key in ['per_device_train_batch_size', 'gradient_accumulation_steps', 'learning_rate']:
    vals = [line.strip() for line in open(tp) if key in line and '=' in line]
    print(key, vals[:4])

## 6. HF Hub 체크포인트 패치 (3번 셀 USE_HF_HUB=True인 경우만)

매 `save_steps`마다 HF Hub에 자동 push → 런타임 꺼져도 복구 가능.

In [ ]:
if USE_HF_HUB and HF_TOKEN:
    tp = '/content/src/train.py'
    s  = open(tp).read()
    hub_snippet = (
        f'push_to_hub            = True,\n'
        f'            hub_model_id           = "{HF_REPO}",\n'
        f'            hub_token              = "{HF_TOKEN}",\n'
        f'            hub_private_repo       = True,\n'
        f'            '
    )
    if 'push_to_hub' not in s:
        s = s.replace('dataset_text_field', hub_snippet + 'dataset_text_field', 1)
        open(tp, 'w').write(s)
        print(f'HF Hub 패치 완료 → {HF_REPO}')
    else:
        print('이미 HF Hub 패치 적용됨')
else:
    print('HF Hub 미사용 — 로컬 저장만 진행')

## 7. 학습 (WEPO-DPO / LoRA)

**모델**: `unsloth/Qwen3-8B-bnb-4bit`, LoRA r=16  
**학습 데이터 구성** (행당 예시 수):
- Step 2 원본 1개
- Step 2 셔플 augmentation × N (real_web ×2)
- Step 1 Grounding 예시 1개 (real_web 전용)

출력 형식: `<think>\n{reasoning}\n</think>\n{json}`  

> 학습 완료 후 바로 셀 8→9→10→11 실행 권장 (런타임 종료 전)

In [ ]:
%cd /content
!python src/train.py

## 8. 학습 결과 확인

In [ ]:
import json, os

metrics_path = '/content/outputs/eval_metrics.json'
if not os.path.exists(metrics_path):
    print('eval_metrics.json 없음 — VALIDATION_MODE=False 이거나 학습 중 오류')
else:
    with open(metrics_path) as f:
        metrics = json.load(f)

    print('=' * 50)
    print('  검증 결과 (eval_metrics.json)')
    print('=' * 50)
    for key, m in metrics.items():
        if not m or m.get('n', 0) == 0:
            continue
        em = m.get('exact_match')
        print(f'\n[{key}]  n={m["n"]}')
        print(f'  op_acc      : {m["op_acc"]:.4f}')
        print(f'  target_acc  : {m["target_id_acc"]:.4f}  ← 핵심 병목')
        print(f'  value_acc   : {m["value_acc"]:.4f}')
        print(f'  exact_match : {em:.4f}  ← 대회 기준' if em else '')

## 9. 추론 → submission.csv

**추론 흐름**:
1. `workflow` HTML → 단발 LLM 배치
2. `real_web` HTML → Step 1 Grounding (action_desc) → 토너먼트 2라운드 → B1 fallback
3. enforce_consistency() → submission.csv

In [ ]:
%cd /content
!python src/inference.py

## 10. 추론 리포트 확인

`artifacts/inference_report.json` — 예측 출처 분포, Grounding Step 1 성공률, 토너먼트 통계, Consistency Guard 수리 횟수.

In [ ]:
import json, os

report_path = '/content/artifacts/inference_report.json'
if not os.path.exists(report_path):
    print('inference_report.json 없음 — 추론을 먼저 실행하세요')
else:
    with open(report_path) as f:
        r = json.load(f)

    total = r.get('total_rows', 1)
    src   = r.get('source', {})
    guard = r.get('consistency_guard', {})
    tstats = r.get('tournament_stats', {})

    print('=' * 55)
    print('  INFERENCE REPORT')
    print('=' * 55)
    print(f'  Total rows        : {total}')

    for k, v in src.items():
        print(f'  {k:<30}: {v["n"]:>5}  ({v["pct"]})')

    think = r.get('thinking_used', {})
    print(f'  thinking_used     : {think.get("n",0):>5}  ({think.get("pct","")})')

    g1 = r.get('grounding_step1', {})
    if g1:
        print(f'  grounding_step1   : done={g1.get("done",0)}, parsed={g1.get("parsed",0)}')

    print('\n  Op 분포:')
    for op, cnt in r.get('op_dist', {}).items():
        print(f'    {op}: {cnt}')

    if guard:
        print('\n  Consistency Guard 수리:')
        for k, v in sorted(guard.items(), key=lambda x: -x[1]['n']):
            print(f'    {k:<38}: {v["n"]:>4}  ({v["pct"]})')

    print('=' * 55)

## 11. 점검 + 다운로드 ← 런타임 꺼지기 전에 반드시 실행

In [ ]:
import pandas as pd
from google.colab import files

df = pd.read_csv('/content/submission.csv')

print('=' * 40)
print(f'  rows          : {len(df)}')
print(f'  op 분포:')
print(df['op'].value_counts(dropna=False).to_string())
print(f'  빈 target_id  : {(df["target_id"].isna() | (df["target_id"].astype(str).str.strip()=="")).sum()}')
print(f'  결측값(전체)  : {df.isnull().sum().sum()}')
print(f'  유효하지 않은 op: {(~df["op"].isin(["CLICK","TYPE","SELECT"])).sum()}')
print('=' * 40)

assert len(df) > 0, '행이 없습니다'
assert df.isnull().sum().sum() == 0, '결측값 존재'
assert (~df['op'].isin(['CLICK','TYPE','SELECT'])).sum() == 0, '잘못된 op 존재'
print('\n검증 통과 — 다운로드 시작')

files.download('/content/submission.csv')

---
## 재연결 후 이어서 학습 (HF Hub 사용한 경우)

런타임 꺼진 뒤 재연결 시: **셀 1~6 재실행 → 이 셀 실행 → 자동 resume**.

In [ ]:
import os, glob

ckpt_dir    = '/content/outputs'
checkpoints = sorted(
    glob.glob(f'{ckpt_dir}/checkpoint-*'),
    key=lambda p: int(p.split('-')[-1])
)

if checkpoints:
    last_ckpt = checkpoints[-1]
    print(f'로컬 체크포인트: {last_ckpt}')
elif USE_HF_HUB:
    last_ckpt = HF_REPO
    print(f'HF Hub에서 resume: {HF_REPO}')
else:
    last_ckpt = None
    print('체크포인트 없음 → 처음부터 학습')

if last_ckpt:
    tp = '/content/src/train.py'
    s  = open(tp).read()
    if 'resume_from_checkpoint' not in s:
        s = s.replace(
            'dataset_text_field',
            f'resume_from_checkpoint = "{last_ckpt}",\n            dataset_text_field',
            1
        )
        open(tp, 'w').write(s)
        print(f'resume 패치 완료 → {last_ckpt}')

%cd /content
!python src/train.py

---
## (선택) 최종 제출 — Full Data 학습

검증 없이 전체 train.csv로 학습. 제출 직전에 실행.

In [ ]:
# FINAL_TRAIN_ON_FULL_DATA = True 패치
import re
tp = '/content/src/train.py'
s  = open(tp).read()
s  = re.sub(r'FINAL_TRAIN_ON_FULL_DATA\s*=\s*False',
            'FINAL_TRAIN_ON_FULL_DATA = True', s)
open(tp, 'w').write(s)
print('FINAL_TRAIN_ON_FULL_DATA = True 패치 완료')

%cd /content
!python src/train.py

---
## (선택) OOF 앙상블

단일 모델 EM 0.75+ 달성 후 최종 제출 전 사용. 학습 시간 3배.  
3-Fold GroupKFold → 각 fold 추론 → 다수결 앙상블.

In [ ]:
%cd /content
!python src/train.py --oof

In [ ]:
%cd /content
!python src/inference.py --ensemble